In [ ]:
from satelliteSegmentation.dataset import AugmentedSubset, SatelliteData, kfold_loaders
from satelliteSegmentation.train import train_model
from satelliteSegmentation.config import Config
from satelliteSegmentation.metrics import segmentation_metrics
from satelliteSegmentation.models import UNet

from torchvision.transforms import v2

import torch
from torch.utils.data import DataLoader

import json
from pathlib import Path

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 64

In [ ]:
data = SatelliteData("data/train")
shape_tf = v2.Compose([
    v2.RandomHorizontalFlip(),
    v2.RandomVerticalFlip(),
    v2.RandomRotation([0, 180], fill=6),
])

color_tf = v2.Compose([
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.1)
])

In [ ]:
loader_kwargs = {
    "num_workers": 2,
    "pin_memory": device == "cuda",
    "persistent_workers": device == "cuda",
}

config = Config(device)

k = 5
results = {}
for fold, train_loader, val_loader in kfold_loaders(
    data,
    k=k,
    batch_size=batch_size,
    shape_transform=shape_tf,
    color_transform=color_tf,
    **loader_kwargs,
):
    print(f"=== Fold {fold + 1}/{k} ===")
    model = UNet(config)
    criterion = torch.nn.CrossEntropyLoss(config.weights, ignore_index=6)
    model, hist = train_model(model, train_loader, val_loader, criterion, config)
    metrics = segmentation_metrics(model, val_loader, config.device, config.num_classes)

    results[f"fold={fold}"] = {
        "hist": hist,
        "metrics": metrics,
    }

In [ ]:
result_folder = Path("result/cross-val")
result_folder.mkdir(parents=True, exist_ok=True)
with open(result_folder / "models.json", "w") as f:
    json.dump(results, f)